In [ ]:
from wiki_dump_reader import Cleaner, iterate
from tqdm import tqdm
import re

In [ ]:
language = "english"

dump_file = f"/mnt/Velocity_Vault/Wiki_Dump/Dump/{language}_dump.xml"
raw_corpus_file = f"/mnt/Velocity_Vault/Wiki_Dump/Corpus/{language}_corpus.txt"
corpus_file = f"/mnt/Velocity_Vault/Wiki_Dump/Corpus/{language}_processed.txt"

In [3]:


def write_corpus(dump, corpus):
    '''
    Processes Wikipedia XML dump and writes cleaned text to corpus file.
    Extracts titles and text, cleans HTML/formatting, removes links,
    and saves with progress tracking using tqdm.
    '''

    cleaner = Cleaner()

    total_pages = sum(1 for _ in iterate(dump))

    with open(corpus, 'w', encoding='utf-8') as output:

        with tqdm(iterate(dump), total=total_pages, unit="page") as pbar:
            for title, text in pbar:

                text = cleaner.clean_text(text)
                cleaned_text, _ = cleaner.build_links(text)
                output.write(title + '\n' + cleaned_text + '\n') # type: ignore

    print(f"Corpus saved to: {corpus}")

    return total_pages

page_count = write_corpus(dump_file,raw_corpus_file)

100%|██████████| 345667/345667 [02:31<00:00, 2286.36page/s]

Corpus saved to: /mnt/Velocity_Vault/Wiki_Dump/Corpus/english_corpus.txt


In [7]:

def clean_line(line):
    '''Cleans English text by converting to lowercase and removing non-alphabetic characters.'''

    line = line.lower()
    cleaned = re.sub(r'[^a-z ]', ' ', line)
    cleaned = re.sub(r' +', ' ', cleaned)
    cleaned = cleaned.strip()

    return cleaned

In [8]:

def preprocess_corpus(source_file, dest_file):
    '''
    Preprocesses corpus file by applying language-specific cleaning.
    Reads source file, cleans each line, removes empty lines,
    and writes continuous text to destination file.
    '''

    total_lines=0

    with open(source_file, 'r', encoding='utf-8') as src:
        total_lines=0
        for _ in src:
            total_lines+=1

    try:
        with open(source_file, 'r', encoding='utf-8') as src:
            with open(dest_file, 'w', encoding='utf-8') as dest:

                first_line = True

                with tqdm(src, total=total_lines, unit="line") as pbar:
                    for line in pbar:

                        cleaned_line = clean_line(line)+" "

                        if not cleaned_line:
                            continue

                        if not first_line:
                            dest.write(' ')
                        else:
                            first_line = False

                        dest.write(cleaned_line)

    except Exception as e:
        print(f"An error occurred: {e}")

    print(f"Processed Corpus saved to: {dest_file}")
    
preprocess_corpus(raw_corpus_file,corpus_file)

100%|██████████| 7433747/7433747 [00:43<00:00, 172830.32line/s]

Processed Corpus saved to: /mnt/Velocity_Vault/Wiki_Dump/Corpus/english_processed.txt
